# Financial Audit GRPO Training — Submission Notebook

**Stack**: Unsloth + TRL GRPOTrainer + OpenEnv financial-audit-env  
**Goal**: Train an LLM via GRPO to detect financial errors (expense overruns, GST mismatches, invoice fraud)  
**Evidence**: Baseline vs. trained comparison on 5 held-out seeds × 4 tasks

### How to run
1. `Runtime → Change runtime type → GPU (T4)`  
2. Add your HF write token to **Colab Secrets** (key icon in sidebar) with key name `HF_TOKEN`  
3. `Runtime → Run all`  

> **Model presets** (edit the Config cell):  
> - Free T4 &nbsp;&nbsp;&nbsp;: `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` (~4 GB VRAM, ~30 min)  
> - HF Credits: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` or `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install trl datasets peft accelerate bitsandbytes pytest httpx huggingface_hub pandas matplotlib

In [ ]:
%cd /content
!rm -rf financial-audit-env
!git clone https://github.com/balloonmann/financial-audit-env.git
%cd /content/financial-audit-env
!pip -q install -e .

In [ ]:
# Add HF_TOKEN to Colab Secrets (key icon in left sidebar) — never paste tokens directly
from google.colab import userdata
from huggingface_hub import login, whoami

token = userdata.get("HF_TOKEN")
login(token=token)
me = whoami()
HF_USER = me["name"]
print(f"Logged in as: {HF_USER}")

In [ ]:
!python -m pytest tests/test_training_pipeline.py -q

## Configuration

| Setting | Free T4 | HF Credits A10G |
|---|---|---|
| MODEL_NAME | Qwen2.5-1.5B | Qwen2.5-7B or Llama-3.1-8B |
| MAX_SEQ_LENGTH | 1536 | 2048 |
| BATCH_SIZE | 1 | 2 |
| NUM_GENERATIONS | 2 | 4 |
| fp16 | True | False (use bf16) |

Change `MODEL_NAME` below to switch preset. Everything else auto-adjusts.

In [ ]:
import os, gc, json, re
import torch
import pandas as pd
import matplotlib.pyplot as plt

# ── Model preset ──────────────────────────────────────────────────────────
# Free T4:    "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"         (~4 GB VRAM)
# HF Credits: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"           (~12 GB VRAM)
#             "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"    (needs HF approval)
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

# ── Hyperparameters ───────────────────────────────────────────────────────
MAX_SEQ_LENGTH  = 1536   # lower to 1024 if OOM on T4
LORA_R          = 16
LORA_ALPHA      = 16
TRAIN_EPOCHS    = 1
BATCH_SIZE      = 1      # keep 1 on T4; raise to 2 on A10G
NUM_GENERATIONS = 2      # GRPO group size; use 4 on A10G
MAX_COMPLETION  = 512
LEARNING_RATE   = 5e-6
LOGGING_STEPS   = 5
SAVE_STEPS      = 50
USE_FP16        = True   # T4 → True; A10G/A100 → False (set bf16=True instead)

# ── Seeds (never change these — reproducibility) ───────────────────────────
TRAIN_SEEDS    = list(range(42, 52))    # 10 seeds used during training
HELD_OUT_SEEDS = list(range(100, 105)) # 5 seeds NEVER seen during training
TASK_IDS = ["expense_audit", "invoice_match", "gst_reconciliation", "fraud_detection"]

# ── Paths ─────────────────────────────────────────────────────────────────
ADAPTER_DIR   = "./grpo-financial-audit-adapter"
ARTIFACTS_DIR = "./artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Config:")
print(f"  model          : {MODEL_NAME}")
print(f"  max_seq_length : {MAX_SEQ_LENGTH}")
print(f"  train_seeds    : {TRAIN_SEEDS}")
print(f"  held_out_seeds : {HELD_OUT_SEEDS}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from financial_audit_env.server.environment import FinancialAuditEnvironment
from financial_audit_env.server.tasks import TASKS
from financial_audit_env.models import AuditAction, Finding
from training.reward import parse_findings_from_text
from training.evaluator import InProcessEvaluator

evaluator = InProcessEvaluator()

def build_prompt(task_id, seed):
    """Build chat-format prompt for a given task+seed."""
    env = FinancialAuditEnvironment()
    obs = env.reset(task_id=task_id, seed=seed)
    task = TASKS[task_id]
    content = (
        "You are a financial auditor. Analyze the documents and output ONLY a JSON array.\n"
        "Each item must have: document_id, error_type, description, confidence (0.0-1.0).\n\n"
        f"TASK: {obs.task_description}\n"
        f"ALLOWED ERROR TYPES: {json.dumps(task['error_types'])}\n"
        f"DOCUMENTS: {json.dumps(obs.documents)[:9000]}\n\n"
        "Output ONLY the JSON array. No explanation."
    )
    return [{"role": "user", "content": content}]

def _norm_conf(x):
    try:
        v = float(x)
    except Exception:
        return 0.7
    if v > 1.0 and v <= 100.0:
        v /= 100.0
    return max(0.0, min(1.0, v))

def run_eval(model, tokenizer, task_ids, seeds, label):
    """Evaluate model on task_ids x seeds. Returns DataFrame."""
    rows = []
    model.eval()
    for tid in task_ids:
        for s in seeds:
            messages = build_prompt(tid, s)
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            completion = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            findings = parse_findings_from_text(completion)
            result = evaluator.evaluate(tid, s, findings)
            rows.append({
                "task_id": tid, "seed": s, "label": label,
                "score": result["score"],
                "weighted_score": result["weighted_score"],
                "precision": result["precision"],
                "recall": result["recall"],
                "num_findings": len(findings),
            })
    return pd.DataFrame(rows)

print("Helpers loaded.")

## Step 1 — Baseline Evaluation (Pre-Training)

Evaluate the **untrained** model on held-out seeds to establish a baseline. These exact seeds will be reused after training to measure improvement.

In [ ]:
print("Loading base model for baseline eval...")
# Load via HF transformers so we can fully free the model before loading Unsloth
HF_BASE_ID = MODEL_NAME.replace("unsloth/", "").replace("-bnb-4bit", "")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
)
base_tok = AutoTokenizer.from_pretrained(HF_BASE_ID, use_fast=True)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_ID, quantization_config=bnb_cfg, device_map={"": 0},
    low_cpu_mem_usage=True, attn_implementation="eager",
)
base_model.config.use_cache = False
print("Model loaded. Running baseline eval...")

baseline_df = run_eval(base_model, base_tok, TASK_IDS, HELD_OUT_SEEDS, label="Baseline")
baseline_df.to_csv(f"{ARTIFACTS_DIR}/baseline_heldout.csv", index=False)

del base_model, base_tok
gc.collect(); torch.cuda.empty_cache()

display(baseline_df)
print(f"\nBaseline mean score: {baseline_df['score'].mean():.4f}")

In [ ]:
baseline_summary = (
    baseline_df.groupby("task_id")[["score","weighted_score","precision","recall"]]
    .mean().reset_index().sort_values("task_id")
)
baseline_summary.to_csv(f"{ARTIFACTS_DIR}/baseline_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(baseline_summary))
ax.bar(x, baseline_summary["score"], color="steelblue", alpha=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(baseline_summary["task_id"], rotation=15)
ax.set_ylabel("Mean Score"); ax.set_title("Baseline Held-out Score by Task")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{ARTIFACTS_DIR}/baseline_score.png", dpi=150, bbox_inches="tight")
plt.show()
display(baseline_summary)

## Step 2 — GRPO Training with Unsloth

The reward function calls the **in-process evaluator** (no HTTP) on every model completion.  
GRPO compares completions within each group of `NUM_GENERATIONS` and reinforces higher-scoring ones.

**Training data**: 4 tasks × 10 seeds = 40 prompts (seeds 42–51)  
**Held-out** (never seen during training): seeds 100–104

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# 1) Load model with Unsloth (4-bit QLoRA)
print(f"Loading {MODEL_NAME} with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# 2) Build training dataset (chat-format prompts + metadata columns)
print("Building training dataset...")
train_rows = []
for tid in TASK_IDS:
    for s in TRAIN_SEEDS:
        train_rows.append({
            "prompt": build_prompt(tid, s),  # list of message dicts (chat format)
            "task_id": tid,
            "seed": s,
        })
train_dataset = Dataset.from_list(train_rows)
print(f"Dataset: {len(train_dataset)} prompts ({len(TASK_IDS)} tasks x {len(TRAIN_SEEDS)} seeds)")

# 3) Reward function — called by GRPOTrainer for each completion group
def reward_fn(completions, task_id, seed, **kwargs):
    rewards = []
    for comp, tid, s in zip(completions, task_id, seed):
        try:
            findings = parse_findings_from_text(comp)
            result = evaluator.evaluate(tid, int(s), findings)
            rewards.append(float(result["score"]))
        except Exception:
            rewards.append(0.01)
    return rewards

# 4) Train
grpo_config = GRPOConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    report_to="none",
    fp16=USE_FP16,
    bf16=not USE_FP16,
)
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

# 5) Save LoRA adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

## Step 3 — Post-Training Evaluation

Run the **trained** model on the same held-out seeds as the baseline.  
These seeds were never in the training set — so this is a clean measure of generalisation.

In [ ]:
# Switch to fast inference mode (Unsloth 2x speedup)
FastLanguageModel.for_inference(model)

print("Running post-training eval on held-out seeds...")
trained_df = run_eval(model, tokenizer, TASK_IDS, HELD_OUT_SEEDS, label="GRPO Trained")
trained_df.to_csv(f"{ARTIFACTS_DIR}/trained_heldout.csv", index=False)

display(trained_df)
print(f"\nTrained mean score: {trained_df['score'].mean():.4f}")

## Step 4 — Before vs. After Comparison

In [ ]:
def summarize(df):
    return df.groupby("task_id")[["score","precision","recall"]].mean()

# Pivot table with delta
pivot = pd.concat([
    summarize(baseline_df).assign(model="Baseline"),
    summarize(trained_df).assign(model="GRPO Trained"),
]).reset_index().pivot_table(index="task_id", columns="model", values="score").round(4)
pivot["delta"] = (pivot["GRPO Trained"] - pivot["Baseline"]).round(4)
display(pivot)

# Side-by-side bar chart
tasks  = list(summarize(baseline_df).index)
x      = range(len(tasks))
width  = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["score", "recall"]):
    base_vals = summarize(baseline_df)[metric].reindex(tasks).values
    trai_vals = summarize(trained_df)[metric].reindex(tasks).values
    ax.bar([i - width/2 for i in x], base_vals, width,
           label="Baseline", color="steelblue", alpha=0.85)
    ax.bar([i + width/2 for i in x], trai_vals, width,
           label="GRPO Trained", color="darkorange", alpha=0.85)
    ax.set_xticks(list(x)); ax.set_xticklabels(tasks, rotation=15)
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"Held-out {metric.capitalize()} — Baseline vs Trained")
    ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACTS_DIR}/before_after_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

delta = trained_df["score"].mean() - baseline_df["score"].mean()
pct   = delta / max(baseline_df["score"].mean(), 1e-6) * 100
print(f"Baseline mean score : {baseline_df['score'].mean():.4f}")
print(f"Trained  mean score : {trained_df['score'].mean():.4f}")
print(f"Delta               : {delta:+.4f}  ({pct:+.1f}%)")

## Step 5 — Backup to Drive + Upload to HuggingFace Hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/financial-audit-artifacts
!cp -r {ARTIFACTS_DIR} /content/drive/MyDrive/financial-audit-artifacts/
!cp -r {ADAPTER_DIR}   /content/drive/MyDrive/financial-audit-artifacts/
print("Backed up to Drive")

In [ ]:
from huggingface_hub import HfApi, upload_folder

api = HfApi()

# Upload trained LoRA adapter
adapter_repo = f"{HF_USER}/financial-audit-grpo-adapter"
api.create_repo(repo_id=adapter_repo, repo_type="model", exist_ok=True)
upload_folder(repo_id=adapter_repo, folder_path=ADAPTER_DIR, repo_type="model")
print(f"Adapter  : https://huggingface.co/{adapter_repo}")

# Upload eval artifacts (CSVs + plots)
artifact_repo = f"{HF_USER}/financial-audit-eval-artifacts"
api.create_repo(repo_id=artifact_repo, repo_type="dataset", exist_ok=True)
upload_folder(repo_id=artifact_repo, folder_path=ARTIFACTS_DIR, repo_type="dataset")
print(f"Artifacts: https://huggingface.co/datasets/{artifact_repo}")

## Results (fill after run)

| Metric | Baseline | GRPO Trained | Delta |
|---|---|---|---|
| Mean Score | — | — | — |
| Mean Precision | — | — | — |
| Mean Recall | — | — | — |

> Plots: `artifacts/before_after_comparison.png`  
> Adapter: HuggingFace link above  
> Full logs: `artifacts/baseline_heldout.csv` and `artifacts/trained_heldout.csv`